In [1]:
import os
import json
from collections import defaultdict
from pathlib import Path
from pprint import pprint
import sys

dirpath_root = Path().resolve().parents[1]
sys.path.append(str(dirpath_root))

In [2]:
def extract_morphology_from_jsons_(cells_dir):
    # Dictionary to store groups of cell types with the same morphology
    morphology_groups = defaultdict(
        lambda: {"pop_names": [], "sections": set(), "section_lists": set()})
    
    # Iterate through all files in the cells/ directory
    for file_name in os.listdir(cells_dir):
        file_path = os.path.join(cells_dir, file_name)
        if file_name.endswith(".json"):
            try:
                with open(file_path, 'r') as f:
                    data = json.load(f)
                
                # Extract sections and section lists
                sections = set(data.get("secs", {}).keys())
                section_lists = set(data.get("secLists", {}).keys())
                
                # Group by morphology (sections and section lists)
                group_key = (frozenset(sections), frozenset(section_lists))
                pop = file_name.split('_')[0]
                morphology_groups[group_key]["pop_names"].append(pop)
                morphology_groups[group_key]["sections"].update(sections)
                morphology_groups[group_key]["section_lists"].update(section_lists)
                
            except Exception as e:
                print(f"Error reading {file_name}: {e}")

    # Convert group keys back to dict for readability
    grouped_morphologies = {}
    for idx, (key, value) in enumerate(morphology_groups.items(), 1):
        grouped_morphologies[f"group_{idx}"] = {
            "pop_names": value["pop_names"],
            "sections": list(value["sections"]),
            "section_lists": list(value["section_lists"]),
        }

    return grouped_morphologies

In [3]:
def _canon_seclists(secLists):
    """
    Return a hashable canonical representation of secLists:
      ( (listName, (sec1, sec2, ...)), ... )
    sorted by listName, and each list sorted.
    """
    if not isinstance(secLists, dict):
        return tuple()

    canon_items = []
    for lname, lst in secLists.items():
        # common patterns:
        #  - ["soma", "dend", ...]
        #  - {"secs": ["soma", ...]}  (seen in some exports)
        if isinstance(lst, dict) and "secs" in lst:
            lst = lst["secs"]

        if lst is None:
            secs = ()
        elif isinstance(lst, (list, tuple, set)):
            secs = tuple(sorted(map(str, lst)))
        else:
            # unexpected type -> keep a stable string
            secs = (str(lst),)

        canon_items.append((str(lname), secs))

    return tuple(sorted(canon_items, key=lambda x: x[0]))


def extract_morphology_from_jsons(cells_dir):
    morphology_groups = defaultdict(lambda: {
        "pop_names": [],
        "sections": set(),
        "section_lists": set(),
        "secLists_signature": None,   # store one exemplar
    })

    for file_name in os.listdir(cells_dir):
        if not file_name.endswith(".json"):
            continue

        file_path = os.path.join(cells_dir, file_name)
        try:
            with open(file_path, "r") as f:
                data = json.load(f)

            sections = set(data.get("secs", {}).keys())
            secLists = data.get("secLists", {}) or {}
            section_lists = set(secLists.keys())

            seclists_sig = _canon_seclists(secLists)

            # NEW: include seclists_sig in grouping key
            group_key = (frozenset(sections), seclists_sig)

            pop = file_name.split("_")[0]
            g = morphology_groups[group_key]
            g["pop_names"].append(pop)
            g["sections"].update(sections)
            g["section_lists"].update(section_lists)
            if g["secLists_signature"] is None:
                g["secLists_signature"] = seclists_sig

        except Exception as e:
            print(f"Error reading {file_name}: {e}")

    grouped = {}
    for idx, (_, value) in enumerate(morphology_groups.items(), 1):
        grouped[f"group_{idx}"] = {
            "pop_names": value["pop_names"],
            "sections": sorted(value["sections"]),
            "section_lists": sorted(value["section_lists"]),
            # optional: keep this to inspect exact list->secs mapping
            "secLists_signature": {
                lname: list(secs) for lname, secs in (value["secLists_signature"] or ())
            },
        }

    return grouped


In [4]:
# Example usage
cells_directory = dirpath_root / 'cells'  # Replace with the path to your cells/ directory
morph_data = extract_morphology_from_jsons(cells_directory)
print(json.dumps(morph_data, indent=4))
morph_data.pop('group_7')

{
    "group_1": {
        "pop_names": [
            "ITP4",
            "PT5B",
            "IT3",
            "IT2",
            "IT6",
            "IT5A",
            "IT5B",
            "CT5B",
            "CT5A",
            "CT6"
        ],
        "sections": [
            "Adend1",
            "Adend2",
            "Adend3",
            "Bdend",
            "axon",
            "soma"
        ],
        "section_lists": [
            "all",
            "apic",
            "apic_lowertrunk",
            "apic_trunk",
            "apic_tuft",
            "apic_uppertrunk",
            "dend_all",
            "proximal"
        ],
        "secLists_signature": {
            "all": [
                "Adend1",
                "Adend2",
                "Adend3",
                "Bdend",
                "soma"
            ],
            "apic": [
                "Adend1",
                "Adend2",
                "Adend3"
            ],
            "apic_lowertrunk": [
               

{'pop_names': ['bkgWeightPops.json'],
 'sections': [],
 'section_lists': [],
 'secLists_signature': {}}

In [5]:
from itertools import zip_longest

def print_groups_as_rows(grouped_morphologies):
    rows = []
    for g, v in grouped_morphologies.items():
        rows.append({
            "group": g,
            "pop_names": ", ".join(v["pop_names"]),
            "sections": ", ".join(v["sections"]),
            "section_lists": ", ".join(v["section_lists"]),
        })

    # column widths
    cols = rows[0].keys()
    widths = {
        c: max(len(c), max(len(r[c]) for r in rows))
        for c in cols
    }

    # header
    header = " | ".join(c.ljust(widths[c]) for c in cols)
    sep = "-+-".join("-" * widths[c] for c in cols)
    print(header)
    print(sep)

    # rows
    for r in rows:
        print(" | ".join(r[c].ljust(widths[c]) for c in cols))


print_groups_as_rows(morph_data)

group   | pop_names                                              | sections                                  | section_lists                                                                         
--------+--------------------------------------------------------+-------------------------------------------+---------------------------------------------------------------------------------------
group_1 | ITP4, PT5B, IT3, IT2, IT6, IT5A, IT5B, CT5B, CT5A, CT6 | Adend1, Adend2, Adend3, Bdend, axon, soma | all, apic, apic_lowertrunk, apic_trunk, apic_tuft, apic_uppertrunk, dend_all, proximal
group_2 | TC, RE, HTC                                            | soma                                      |                                                                                       
group_3 | NGF, TI, TI, TI                                        | dend, soma                                | all, dend_all, proximal                                                               
group_4 | 

In [6]:
import pandas as pd

df = pd.DataFrame([
    {
        "group": g,
        "pop_names": ", ".join(v["pop_names"]),
        "sections": ", ".join(v["sections"]),
        "section_lists": ", ".join(v["section_lists"]),
    }
    for g, v in morph_data.items()
])

#print(df.to_string(index=False))
df


,group,pop_names,sections,section_lists
0,group_1,"ITP4, PT5B, IT3, IT2, IT6, IT5A, IT5B, CT5B, C...","Adend1, Adend2, Adend3, Bdend, axon, soma","all, apic, apic_lowertrunk, apic_trunk, apic_t..."
1,group_2,"TC, RE, HTC",soma,
2,group_3,"NGF, TI, TI, TI","dend, soma","all, dend_all, proximal"
3,group_4,ITS4,"Adend1, Adend2, Adend3, Bdend, soma","all, apic, apic_lowertrunk, apic_trunk, apic_t..."
4,group_5,"SOM, PV","axon, dend, soma","all, dend_all, proximal"
5,group_6,VIP,"ori1, ori2, rad1, rad2, soma","all, dend_all, proximal"


In [10]:
def morph_groups_to_seclist_table(morph_data, joiner=", "):
    all_seclists = sorted({
        sl for g in morph_data.values()
        for sl in g.get("secLists_signature", {}).keys()
    })

    rows = []
    idx = []

    for group_name, g in morph_data.items():
        sig = g.get("secLists_signature", {}) or {}

        row = {}
        row["pop_names"] = ", ".join(g.get("pop_names", []))  # NEW column

        for sl in all_seclists:
            secs = sig.get(sl)
            row[sl] = "" if not secs else joiner.join(secs)

        rows.append(row)
        idx.append(group_name)

    df = pd.DataFrame(rows, index=idx)
    df.index.name = "pop_group"

    # ensure pop_names is the first column
    cols = ["pop_names"] + [c for c in df.columns if c != "pop_names"]
    df = df[cols]

    return df

df = morph_groups_to_seclist_table(morph_data)
df

,pop_names,all,apic,apic_lowertrunk,apic_trunk,apic_tuft,apic_uppertrunk,dend_all,proximal
pop_group,,,,,,,,,
group_1,"ITP4, PT5B, IT3, IT2, IT6, IT5A, IT5B, CT5B, C...","Adend1, Adend2, Adend3, Bdend, soma","Adend1, Adend2, Adend3",Adend1,"Adend1, Adend2",Adend3,Adend2,"Adend1, Adend2, Adend3, Bdend","Adend1, Bdend, soma"
group_2,"TC, RE, HTC",,,,,,,,
group_3,"NGF, TI, TI, TI","dend, soma",,,,,,dend,"dend, soma"
group_4,ITS4,"Adend1, Adend2, Adend3, Bdend, soma","Adend1, Adend2, Adend3",Adend1,"Adend1, Adend2",Adend3,Adend2,"Adend1, Adend2, Adend3, Bdend","Adend1, Bdend, soma"
group_5,"SOM, PV","dend, soma",,,,,,dend,"dend, soma"
group_6,VIP,"ori1, ori2, rad1, rad2, soma",,,,,,"ori1, ori2, rad1, rad2","ori1, rad1, soma"
